# 模組 5：用遺傳演算法（GA）挑投資組合 ＋ 揪過度最佳化

用現成的 GA 套件 `PyGAD`，即時抓真實的 0050 成分股股價，讓它自動找一組「權重」湊出夏普比（划算程度）最高的投資組合；再切樣本內／外，看漂亮的回測拿到「沒看過的未來」還剩幾分——這就是**過度最佳化**。

> 需要網路（要即時抓股價）。不需 API / Ollama，純本地算。

> ## ⚠️ 金融免責（先讀）
> 今天用的是**真實的 0050 成分股**，但整堂課只示範「怎麼做最佳化、怎麼揪過度最佳化」，**不是選股、不是投資建議**。GA 挑出來的投組漂不漂亮，看的是回測數字，不代表未來會賺。真正的投資決策要由人判斷（human-in-the-loop）。今天最該記住的，反而是「**別信漂亮的回測**」。

## 🟦 A 段：今天要做什麼

一句話：**找一組權重，讓一籃子股票的「夏普比」最高**。

- **投資組合**＝同時買好幾檔、各給一個**權重**（各買多少比例）。
- **夏普比**＝這籃子「賺得穩不穩」的分數，越高越划算。
- **GA（遺傳演算法）** ＝模仿生物演化，一代一代自動試出好權重的搜尋法；內部四步（算分／選擇／交配／突變）點到即止，細節看投影片與課堂講解。
- **PyGAD**＝現成的 GA 輪子，我們不用自己刻，只要把「要最大化的分數」餵給它。

> 概念與生活類比在課堂講解，這裡直接動手。

## 🟩 B 段：抓資料 → 算報酬 → 讓 PyGAD 找最佳投組

先裝套件。已裝好的會直接跳過；裝不起來看 README。

In [ ]:
# 【格1】裝套件
!pip install -q pygad yfinance    # 💡 -q＝安靜安裝，少洗版；pygad＝現成 GA 輪子、yfinance＝抓股價

In [ ]:
# import 今天要用的套件
import numpy as np, pandas as pd, yfinance as yf, pygad

### 先抓一檔看看：yfinance 怎麼用

抓一整籃之前，先拿**台積電（代號 `2330.TW`）** 一檔試手，看 `yf.download()` 回什麼。台股代號後面要加 `.TW`。

In [ ]:

# 先抓「台積電」一檔，熟悉 yf.download 的用法
# TODO：抓單一檔股票。函式名跟下面抓一整籃時同一個；第一個參數放「單檔代號字串」（台積電＝2330.TW）
stock_2330 = yf.____("____", period="4y", interval="1d", auto_adjust=True, progress=False)
stock_2330.tail()


### 0050 成分股清單

熟悉單檔後，換成一整籃 0050 成分股的代號。

> 🎯 這是大型權值股清單；**正式成分股以當期公告為準**，清單會換。抓不到資料的（下市、改代號）後面會自動剔除，不用手動改。

In [ ]:
# 0050 成分股（大型權值股；正式清單以當期公告為準，抓不到的自動剔除）
tickers = ["2330","2317","2454","2308","2382","2881","2882","2412","2891","2303",
           "3711","2886","2884","1301","1303","2002","2207","3008","2357","2379",
           "3034","2395","2345","2890","2892","5880","2885","1216","2603","2609",
           "2615","3037","3231","2356","4938","6505","1101","2409","3045","2327",
           "2408","1326","2474","6415","3661"]
tw = [t + ".TW" for t in tickers]    # 💡 yfinance 要「代號.TW」才認得台股（上市）

### 即時抓近 4 年股價

跟 yfinance 要一整籃近 4 年的每日收盤價，然後**分兩步清乾淨**：先丟「整欄都沒資料」的股票，再丟「還有缺值」的天。

> 🎲 **即時抓，每次跑到的資料都略不同**——後面所有數字看趨勢、不對精確值。
> 🌐 抓股價要連網；跑很慢或抓到 0 檔多半是網路擋 yfinance，重跑或看 README。

In [ ]:
# 【格3】即時抓近 4 年股價
raw = yf.download(tw, period="4y", interval="1d", auto_adjust=True, progress=False)  # 💡 auto_adjust＝還原除權息，報酬才不會失真
close = raw["Close"] if isinstance(raw.columns, pd.MultiIndex) else raw    # 💡 只抓到 1 檔時欄位長相不同，這行兩種都接得住
close    # 先看一眼原始長相（有些欄可能整欄 NaN、有些天有缺值）

In [ ]:

# 第 1 步：丟掉「整欄都是空值」的股票（近 4 年完全沒資料的）
# TODO：用 pandas 丟缺值的方法，指定 axis=1（看欄）、how="all"（整欄全空才丟）
step_1 = close.____(axis=1, how="all")
step_1


In [ ]:

# 第 2 步：再丟掉「還有缺值的那幾天」→ 只留全程都有資料的乾淨表
# TODO：跟上一格同一個方法，這次不給參數（預設丟「含任何缺值的天」）
step_2 = step_1.____()
step_2


In [ ]:
close = step_2
print("抓到", close.shape[1], "檔、", len(close), "個交易日")
close.tail()

### 算每日報酬

把股價轉成「每天漲跌幾 %」——這就是等一下算夏普比、給 GA 最大化的原料。

In [ ]:

# 算每日報酬
# TODO：把「股價」換成「每天漲跌幾 %」（今天相對昨天的變化比例）。第一天沒昨天可比，接 .dropna() 丟掉
rets = close.____().dropna()
rets


In [ ]:
N = rets.shape[1]
print("資產數：", N)

### 讓 PyGAD 找最佳投組（先用最近 60 天當範例）

先拿最近 60 天（約一季／季線）練手，建立 PyGAD 的用法，看它挑出的投組夏普比多高。

> 🎯 **60 天＝季線**，技術分析常用的窗。
> 🕹️ 我們只餵「要最大化的分數」（夏普比）給 PyGAD，**GA 內部怎麼演化是黑盒子**——原理看投影片，這裡只看結果。

In [ ]:

WIN = 60
# TODO：取最近 WIN（60）個交易日。提示：pandas 取「最後幾列」的方法
recent = rets.____(WIN)
recent


In [ ]:

def sharpe(weights, R):
    """夏普比（年化）= 投組平均報酬 / 報酬標準差 × √252。越高越划算。"""
    w = np.clip(np.asarray(weights, float), 0, None)   # 負權重壓成 0＝不放空、只做多
    if w.sum() == 0:
        return 0.0
    w = w / w.sum()                                    # 正規化成合法權重（加總＝1）
    port = R.values @ w                                # 每一天的投組報酬＝各股報酬 × 各自權重再加總
    return 0.0 if port.std() == 0 else port.mean() / port.std() * np.sqrt(252)

def fitness_func(ga_instance, solution, solution_idx):
    # TODO：回傳「這組權重」在「最近 60 天」上的夏普比 → 呼叫上面的 sharpe()
    #   第一個參數＝這代要評分的權重（PyGAD 傳進來的那個變數）
    #   第二個參數＝要用哪段資料算分（上面剛定義的那段 60 天資料變數）
    return sharpe(____, ____)

ga = pygad.____(num_generations=120,          # TODO：PyGAD 建立 GA 的類別（大寫兩個字母）
              num_parents_mating=12,
              fitness_func=fitness_func,
              sol_per_pop=40,
              num_genes=N,
              gene_space={'low': 0.0, 'high': 1.0},
              gene_type=float,
              random_seed=83,
              mutation_percent_genes=20,
              suppress_warnings=True)
ga.run()


### GA 跑完了，看它找到什麼

`ga.best_solution()` 回傳三樣：**最佳解、最佳分數、索引**。先把「最佳解」印出來看看它長什麼樣子。

In [ ]:

# TODO：取出 GA 找到的最佳解。提示：PyGAD 取最佳解的方法，回傳（解, 分數, 索引）
best, best_fit, _ = ga.____()
best    # 這是 GA 找到的一組「原始權重」——還沒正規化，可能有負、加起來也不是 1


In [ ]:
best_w = np.clip(best, 0, None)
best_w = best_w / best_w.sum()   # 💡 把原始權重壓非負＋正規化成合法投組（加總＝1）
print("GA 找到的最佳投組夏普比：", round(sharpe(best_w, recent), 2))

In [ ]:
# 看它押最重的前 5 檔
top5 = np.argsort(best_w)[::-1][:5]    # 💡 argsort 由小到大排索引，[::-1] 反轉成由大到小，取前 5
for i in top5:
    print(f"  {rets.columns[i]}  權重 {best_w[i]:.1%}")

> 說明：這 5 檔只是「這 60 天回測最划算」的組合，**不是推薦買它們**。下一段就會看到，換一段時間考它，這組合可能就不靈了。

## 🟧 C 段：切樣本內／外 — 揪過度最佳化

前面 GA 在同一段資料裡「又找答案又打分」，當然漂亮。真正的考驗是：**用一段資料找出的投組，拿到另一段沒看過的時間，還剩幾分？**

- **樣本內**＝GA 拿來找權重的那段（念書範圍）。
- **樣本外**＝另一段沒看過的時間（考試）。
- **過度最佳化**＝把回測那段的雜訊也當規律硬記，換段時間就縮水。

### 先複習：用「序號」切一段資料

等一下要用序號把資料切成「前一段／後一段」。先用一個小 list 熱身，看序號切片 `[起:迄]` 怎麼運作（含頭不含尾）。

In [ ]:
list_a = [10, 11, 12, 13, 14, 15, 16, 17, 18]
print("list_a[1:3] =", list_a[1:3])   # 從第 1 個到第 3 個「之前」（含頭不含尾）→ 11, 12
print("list_a[3:]  =", list_a[3:])    # 從第 3 個到最後 → 13,14,15,16,17,18
print("倒數 3 個 list_a[-3:] =", list_a[-3:])

### 按時間切成「更早 60 天（樣本內）」和「最近 60 天（樣本外）」

`split` ＝總天數減 60，當作分界。樣本內＝`split` 往前 60 天，樣本外＝`split` 到最後。**注意是「時序切」**：較早的念書、較晚的考試，不可隨機打散（那會偷看未來）。

In [ ]:

# 【格6】切樣本內 / 樣本外（都是 60 天）
split = len(rets) - WIN
# TODO：樣本內＝「更早的 60 天」＝從 split 往前數 WIN 天、到 split（用 split-WIN 和 split）
rets_in  = rets.iloc[____:____]
# TODO：樣本外＝「最近的 60 天」＝從 split 到最後
rets_out = rets.iloc[____:]
print(f"樣本內 {len(rets_in)} 天 → 樣本外 {len(rets_out)} 天")


### 主示範：GA 只在樣本內找 → 樣本外考 → 對比「平均分配」

規矩改對：GA **只准看樣本內**找權重，再拿樣本外考。同時擺一個「平均分配」（不動腦、每檔一樣多）當基準。

> 👀 **看兩件事：** ①GA 的樣本內 vs 樣本外差多少（縮水＝過度最佳化）；②樣本外時，GA 有沒有贏過「平均分配」。
> ⏳ 這格要跑一次完整 GA，會等幾秒。

In [ ]:
# 【格7】把「用一段資料找最佳權重」包成函式（樣本內、掃窗都會用到）
def ga_best_weights(train):    # 💡 餵一段資料 train，回傳「只看這段找到的」最佳權重
    def fit(ga_i, sol, idx): return sharpe(sol, train)   # 💡 只用 train 打分，樣本外完全沒參與訓練
    g = pygad.GA(num_generations=120, num_parents_mating=12, fitness_func=fit,
                 sol_per_pop=40, num_genes=N, gene_space={'low':0.0,'high':1.0},
                 gene_type=float, random_seed=83, mutation_percent_genes=20,
                 suppress_warnings=True)
    g.run()
    w = np.clip(g.best_solution()[0], 0, None); return w / w.sum()

In [ ]:

# TODO：用上面的函式，讓 GA 只吃「樣本內」找最佳權重（函式名就是上一格 def 的那個）
w_ga = ____(rets_in)
eq   = np.ones(N) / N          # 平均分配（不動腦，每檔一樣多）＝最笨的對照組
print(f"{'策略':<12}{'樣本內':>10}{'樣本外':>10}")
print(f"{'GA 最佳':<12}{sharpe(w_ga, rets_in):>10.2f}{sharpe(w_ga, rets_out):>10.2f}")
print(f"{'平均分配':<12}{sharpe(eq, rets_in):>10.2f}{sharpe(eq, rets_out):>10.2f}")


### 怎麼讀這張表（即時資料，你那次可能不同）

**保證會看到的主軸：** GA 的**樣本內分數漂亮 → 樣本外縮水**。GA 把樣本內的雜訊也硬背了，換段時間就打回原形，這就是過度最佳化。

**很常見的彩蛋：** GA 的**樣本外還輸給「平均分配」**——辛苦最佳化半天，不如每檔買一樣多。

**萬一你這次樣本外沒縮水：** 代表這段真實市場訊號夠強、GA 這次沒過度最佳化。但——你敢賭下次也這樣嗎？這正是為什麼**永遠要看樣本外**。下面的掃窗作業（尤其 1 個月的短窗）幾乎一定會讓你看到崩給你看。

> 🔗 同一個道理：課程 1 的 Precision／Recall、模組 3 的 k、模組 4 的樹深、今天的過度最佳化——都是「別只看訓練分數」的同一把誠實的尺。

## 🟫 D 段：收尾 — 一把貫穿全課的誠實的尺

今天把「數據路」三招收官：模組 3 KNN、模組 4 決策樹、模組 5 GA 最佳化。串起來就是一句：**任何漂亮的模型／回測，都要拿「沒看過的資料」再驗一次。**

```mermaid
flowchart LR
    A[課程1<br/>Precision/Recall] --> B[模組3<br/>選 k]
    B --> C[模組4<br/>樹深]
    C --> D[模組5<br/>GA 過度最佳化]
    D --> E[模組10<br/>回測 ≠ 實盤]
```

> 🔗 同一把尺，量到底：訓練分數再漂亮，樣本外／實盤才算數。

> ## 🚫 收尾免責（再一次，加重）
> 今天用真實 0050 只為了示範最佳化與過度最佳化，**不是選股、不是投資建議**。GA 挑的投組在回測漂亮、樣本外就縮水，本身就是「別信漂亮回測」最好的反例。真上場前，一切要人來判斷（human-in-the-loop），沒有一組權重是「照抄就會賺」。
>
> 🎓 **iPAS / AI-901 接點：** 最佳化問題、過度最佳化 vs 過擬合、樣本內外驗證、模型評估、human-in-the-loop 的必要性。

## 📝 回家作業：掃不同時間窗，看誰崩最兇

上面用 60 天。作業把窗換成 1 個月 / 3 個月 / 6 個月 / 1 年，各跑一次 GA，比樣本內 → 樣本外崩多少，也對比平均分配。

> 🎯 **要驗證的猜想：窗越短，越容易過度最佳化**（短窗雜訊多、GA 更容易硬背）。
> 下面是完整版參考解，先自己想「哪個窗崩最兇」再看跑出來的結果。

In [ ]:

# 【格8・回家作業】掃時間窗：窗越短越容易過度最佳化
for name, W in {"1m":21, "3m":63, "6m":126, "1y":252}.items():
    train = rets.iloc[____:split]     # TODO：樣本內起點＝從 split 往前數 W 天（照 60 天窗的寫法，把 WIN 換成 W）
    w = ga_best_weights(train)
    gi, go, eo = sharpe(w, train), sharpe(w, rets_out), sharpe(eq, rets_out)
    print(f"{name:>4} 窗：樣本內 {gi:5.2f} → 樣本外 {go:5.2f}（崩 {go-gi:+5.2f}）  平均分配樣本外 {eo:5.2f}")


### 📌 參考解結論

即時資料每次不同，但趨勢通常是：

- **1 個月窗崩最兇**——短窗雜訊多，GA 把雜訊當規律硬背，樣本外幾乎一定大縮水，還常輸平均分配。
- **窗拉長（6 個月、1 年）縮水通常收斂**——資料多，GA 比較沒空間過度最佳化。

**一句帶走：窗越短、越容易過度最佳化。** 這也是為什麼看回測不能只挑一段漂亮的時間——換段時間、拉長時間再驗，才知道是不是真本事。

## 🛟 附錄：套件驗證（不需網路）

某格連不上 yfinance 時，先跑這個確認 **PyGAD 本身會動**（玩具問題：找 6 個數字讓總和最大）。這幾行能跑，套件層就沒問題，剩下就是網路能不能連 yfinance。

In [ ]:
!pip install -q pygad
import pygad
def fit(ga, sol, idx): return sum(sol)                 # fitness = 總和（越大越好）
g = pygad.GA(num_generations=30, num_parents_mating=4, fitness_func=fit,
             sol_per_pop=10, num_genes=6, gene_space={'low':0.0,'high':1.0},
             gene_type=float, random_seed=83, suppress_warnings=True)
g.run()
print("最佳解:", g.best_solution()[0].round(2), "總和:", round(g.best_solution()[1], 2))